[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/09_higher_order_grad_solution.ipynb)

# 🟡 Solution: Hessian with jacfwd(jacrev(f))

*JAX Fundamentals · Medium*

Reference implementation. Try it yourself in `09_higher_order_grad.ipynb` first.

---
Return the **Hessian matrix** of a scalar function at a point.

$$H_{ij} = \frac{\partial^2 f}{\partial x_i\, \partial x_j}$$

Given `f: (D,) -> scalar` and `x: (D,)`, return the `(D, D)` matrix of second
derivatives.

### Rules
- Compose `jax.jacfwd` and `jax.jacrev` — do not use `jax.hessian`
- Use the **forward-over-reverse** order: `jacfwd(jacrev(f))`
- Must work for any `D` and stay `jit`-able

### Signature
```python
def hessian_matrix(f, x):  # -> (D, D)
    ...
```

### Why the order matters
This is the real question behind the problem. For `f: R^D -> R`:

- `jacrev` costs **one** pass regardless of `D` (one output), so it is the right
  choice for the inner gradient
- `jacfwd` costs **one pass per input**, and it is applied to the `D`-dimensional
  gradient, giving `D` passes total
- Doing it the other way, `jacrev(jacfwd(f))`, computes the inner Jacobian in
  `D` forward passes and then reverse-differentiates that whole thing — same
  asymptotics but a much larger tape and more memory

Reverse mode is cheap in the number of **outputs**; forward mode is cheap in the
number of **inputs**. Being able to say that out loud is the point.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def hessian_matrix(f, x):
    # Inner jacrev: one reverse pass for the scalar output -> (D,) gradient.
    # Outer jacfwd: D forward passes over that gradient -> (D, D) Hessian.
    return jax.jacfwd(jax.jacrev(f))(x)

In [ ]:
# 🔍 Verify
import jax.numpy as jnp

# f(x) = x0^2 + 3*x0*x1 + 2*x1^2
# H = [[2, 3], [3, 4]]
f = lambda x: x[0] ** 2 + 3 * x[0] * x[1] + 2 * x[1] ** 2

print(hessian_matrix(f, jnp.array([1.0, 1.0])))

# A quadratic form x^T A x has Hessian A + A^T, independent of x:
A = jnp.array([[1.0, 2.0], [0.0, 3.0]])
q = lambda x: x @ A @ x
print(hessian_matrix(q, jnp.array([5.0, -7.0])))   # == A + A.T

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("higher_order_grad")